In [ ]:
import numpy as np
import scipy.stats as stats
from scipy.integrate import quad
from scipy.optimize import brentq

def log_normal_ppf(a, sigma, q):
    return stats.lognorm.ppf(q, sigma, scale = np.exp(a))

def eq_for_c(c, a, sigma, eps, var):
    lower = 1 - c * eps
    upper = 1
    integral, _ = quad(lambda q: log_normal_ppf(a, sigma, q), lower, upper)
    left_side = (1 / (c * eps)) * integral
    return left_side - var

def find_c(a, sigma, eps, var):
    c_sol = brentq(eq_for_c, 1, 1 / eps, args=(a, sigma, eps, var))
    return c_sol

a, sigma = 0, 1
eps = 1e-13
var = log_normal_ppf(a, sigma, 1 - eps)

c = find_c(a, sigma, eps, var)
print(f"c = {c}")


In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats
from scipy.integrate import quad
from scipy.optimize import brentq

def log_normal_isf(sigma, q):
    return stats.lognorm.isf(q, sigma)

def eq_for_c(c, sigma, eps, var):
    def integrand(t):
        p = c * eps * t
        return log_normal_isf(sigma, p)
    integral, _ = quad(integrand, 0, 1, epsabs=1e-14, epsrel=1e-14, limit=1000)
    return integral - var

def find_c(sigma, eps, var):
    c_sol = brentq(eq_for_c, 1, 1 / eps, args=(sigma, eps, var))
    return c_sol

sigma2 = 1
eps = 1e-1
var = log_normal_isf(math.sqrt(sigma2), eps)

num_points_linear = 40
num_points_log = 50
total_points = num_points_linear + num_points_log

eps_linear = np.linspace(1e-1, 1e-5, 40, endpoint=False)

eps_log = np.logspace(-2.585, -100, 50)

eps_values_combined = np.concatenate((eps_linear, eps_log))

x_values = 1.0 - eps_values_combined

c_results = []

print(f"Calculating c for {total_points} combined values of eps:")

count_nan = 0
for i, eps in enumerate(eps_values_combined):

    var = log_normal_isf(math.sqrt(sigma2), eps)
    c = find_c(math.sqrt(sigma2), eps, var)

    c_results.append(c)

    if (i + 1) % (total_points // 10) == 0:
        print(f"Processed {i+1}/{total_points} points...")

print(f"Calculation finished. {count_nan} points resulted in NaN for c.")

x_values_plot = np.array(x_values)
c_results_plot = np.array(c_results)

valid_indices = ~np.isnan(c_results_plot)
x_plot = x_values_plot[valid_indices]
c_plot = c_results_plot[valid_indices]


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(10, 6))

if len(x_plot) > 0:

    plt.plot(x_plot, c_plot, marker='', linestyle='-', markersize=4, label=r'$c(\varepsilon)$')
else:
    print("Warning: No valid points to plot.")

euler_e = math.e
plt.axhline(y=euler_e, color='red', linestyle='-', linewidth=1.5,
            label=fr'$y = e \approx {euler_e:.5f}$')

plt.rc('text', usetex=False)
plt.xlabel(r'$1 - \varepsilon$', fontsize=14)
plt.ylabel(r'$\Pi_\varepsilon(X)$', fontsize=14)

plt.title(r'', fontsize=16)

if len(c_plot) > 0:

     min_c_val = np.nanmin(c_plot)
     max_c_val = np.nanmax(c_plot)

     if np.isfinite(min_c_val) and np.isfinite(max_c_val) and np.isfinite(euler_e):
         plot_min_y = min(min_c_val * 0.995, euler_e * 0.99)
         plot_max_y = max(max_c_val * 1.005, euler_e * 1.01)
         y_range = plot_max_y - plot_min_y

         if np.isfinite(y_range) and y_range > 1e-9:
             plt.ylim(plot_min_y - 0.01*y_range, plot_max_y + 0.01*y_range)
         elif np.isfinite(plot_min_y) and np.isfinite(plot_max_y):
              plt.ylim(plot_min_y - 0.1, plot_max_y + 0.1)

plt.grid(True)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
import math
import numpy as np
import scipy.stats as stats
from scipy.integrate import quad
from scipy.optimize import brentq

def log_normal_isf(sigma, q):
    return stats.lognorm.isf(q, sigma)

def eq_for_c(c, sigma, eps, var):
    def integrand(t):
        p = c * eps * t
        return log_normal_isf(sigma, p)
    integral, _ = quad(integrand, 0, 1, epsabs=1e-14, epsrel=1e-14, limit=1000)
    return integral - var

def find_c(sigma, eps, var):
    c_sol = brentq(eq_for_c, 1, 1 / eps, args=(sigma, eps, var),maxiter=10000)
    return c_sol

sigma2_values = [0.04, 0.25, 1.0]
colors = ['blue', 'green', 'purple']

num_points_linear = 40
num_points_log = 50
total_points = num_points_linear + num_points_log

eps_linear = np.linspace(1e-1, 1e-5, 40, endpoint=False)
eps_log = np.logspace(-2.585, -20, 50)

eps_values_combined = np.concatenate((eps_linear, eps_log))

x_values = 1.0 - eps_values_combined

all_c_results = {}
all_valid_c_values_for_ylim = []

print("Starting calculations...")

for sigma2 in sigma2_values:
    sigma = math.sqrt(sigma2)
    print(f"\nCalculating for sigma^2 = {sigma2} (sigma = {sigma:.4f})")
    c_results = []
    count_nan = 0

    for i, eps in enumerate(eps_values_combined):

        var = log_normal_isf(sigma, eps)

        c = find_c(sigma, eps, var)
        if np.isnan(c):
            count_nan += 1

        c_results.append(c)

        if (i + 1) % (total_points // 10) == 0:
             print(f"  Processed {i+1}/{total_points} points for sigma^2 = {sigma2}...")

    all_c_results[sigma2] = np.array(c_results)
    print(f"Calculation finished for sigma^2 = {sigma2}. {count_nan} points resulted in NaN for c.")

    valid_c = c_results
    if len(valid_c) > 0:
        all_valid_c_values_for_ylim.extend(valid_c)


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(10, 6))

for i, sigma2 in enumerate(sigma2_values):
    c_results_plot = all_c_results[sigma2]
    color = colors[i]

    valid_indices = ~np.isnan(c_results_plot)
    x_plot = x_values[valid_indices]
    c_plot = c_results_plot[valid_indices]

    if len(x_plot) > 0:

        plt.plot(x_plot, c_plot, marker='', linestyle='-', linewidth=1.5,
                 color=color, label=fr'$\sigma^2 = {sigma2}$')
    else:
        print(f"Warning: No valid points to plot for sigma^2 = {sigma2}.")

euler_e = math.e
plt.axhline(y=euler_e, color='red', linestyle='-', linewidth=1.5,
            label=fr'$y = e \approx {euler_e:.5f}$')

try:
    plt.rc('text', usetex=False)
    xlabel_text = r'$1 - \varepsilon$'
    ylabel_text = r'$\Pi_\varepsilon(X)$'

except RuntimeError:
    print("LaTeX not found or not configured. Using standard Matplotlib rendering.")
    plt.rc('text', usetex=False)
    xlabel_text = '1 - epsilon'
    ylabel_text = 'c(epsilon)'
    title_text = 'Value c(epsilon) for Log-Normal Distribution vs. 1-epsilon'

plt.xlabel(xlabel_text, fontsize=14)
plt.ylabel(ylabel_text, fontsize=14)

if all_valid_c_values_for_ylim:
     min_c_val = np.min(all_valid_c_values_for_ylim)
     max_c_val = np.max(all_valid_c_values_for_ylim)

     if np.isfinite(min_c_val) and np.isfinite(max_c_val) and np.isfinite(euler_e):

         plot_min_y = min(min_c_val, euler_e)
         plot_max_y = max(max_c_val, euler_e)
         y_range = plot_max_y - plot_min_y

         if np.isfinite(y_range) and y_range > 1e-9:
             padding = 0.05 * y_range
             plt.ylim(plot_min_y - padding, plot_max_y + padding)
         else:
             plt.ylim(plot_min_y - 0.1, plot_max_y + 0.1)

plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

print("\nPlotting complete.")


In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats
from scipy.integrate import quad
from scipy.optimize import brentq

def log_normal_isf(sigma, q):
    return stats.lognorm.isf(q, sigma)

def eq_for_c(c, sigma, eps, var):
    def integrand(t):
        p = c * eps * t
        return log_normal_isf(sigma, p)
    integral, _ = quad(integrand, 0, 1, epsabs=1e-14, epsrel=1e-14, limit=1000)
    return integral - var

def find_c(sigma, eps, var):
    c_sol = brentq(eq_for_c, 1, 1 / eps, args=(sigma, eps, var), maxiter=10000)
    return c_sol

sigma2 = 0.25
eps = 1e-308
var = log_normal_isf(math.sqrt(sigma2), eps)

c = find_c(math.sqrt(sigma2), eps, var)
print(c)
